In [ ]:
# In[1]:

import pandas as pd

# load metric.csv
metric = pd.read_csv('metric.csv')

# parse timestamp as UTC datetime per rules
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# ensure value is numeric
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# aggregate across the full series (no time filtering)
agg = (
    metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        count='count',
        p50=lambda x: x.quantile(0.5),
        p95=lambda x: x.quantile(0.95),
        max='max'
    )
    .reset_index()
)

# sort by count descending and keep a compact top 20
result = agg.sort_values('count', ascending=False).head(20)

result

```
Out[1]:
```
summary = (
    "I computed count, P50, P95 and max for each cmdb_id + kpi_name across the entire metric.csv.\n"
    "All top 20 combinations have 25 data points each (count=25).\n"
    "Notable observations from the top results:\n"
    "- adservice.diskio: P95 ≈ 75,180, max ≈ 93,419 (much larger than other diskio values).\n"
    "- adservice.mem: P50 ≈ 112,639,600, P95 ≈ 112,811,300, max ≈ 112,834,300 (large memory footprint).\n"
    "- checkoutservice.latency-90: P50 ≈ 0.223 s, P95 ≈ 0.238 s, max ≈ 0.282 s (higher latency than cartservice/adservice).\n"
    "- cartservice.latency-90: P95 ≈ 0.00568 s, max ≈ 0.00630 s (low latency overall).\n"
    "- socket metrics: checkoutservice shows socket values of 9 (p50/p95/max = 9), adservice and cartservice show ~4.\n"
    "These summaries use the full available series (no time filtering)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  count           p50           p95           max
0         adservice         cpu     25  6.944591e-01  7.596033e-01  7.792910e-01
1         adservice      diskio     25  6.231735e+04  7.517982e+04  9.341964e+04
2         adservice  latency-50     25  3.003831e-03  3.028602e-03  3.031522e-03
3         adservice  latency-90     25  4.615787e-03  4.675922e-03  4.679990e-03
4         adservice         mem     25  1.126396e+08  1.128113e+08  1.128343e+08
5         adservice      socket     25  4.000000e+00  4.000000e+00  4.233333e+00
6         adservice    workload     25  1.446215e+01  1.533943e+01  1.562114e+01
7       cartservice         cpu     25  2.022764e+00  2.523032e+00  2.548629e+00
8       cartservice      diskio     25  0.000000e+00  0.000000e+00  0.000000e+00
9       cartservice  latency-50     25  3.147306e-03  3.221937e-03  3.274059e-03
10      cartservice  latency-90     25  4.865151e-03  5.684042e-03  6.300198e-03
11      cartservice         mem     25  4.455117e+07  4.463935e+07  4.464551e+07
12      cartservice      socket     25  4.000000e+00  4.000000e+00  4.000000e+00
13      cartservice    workload     25  2.414342e+01  2.675402e+01  2.705236e+01
14  checkoutservice         cpu     25  4.048152e-01  5.267428e-01  5.511535e-01
15  checkoutservice      diskio     25  0.000000e+00  0.000000e+00  0.000000e+00
16  checkoutservice  latency-50     25  9.639017e-02  1.172605e-01  1.226884e-01
17  checkoutservice  latency-90     25  2.230191e-01  2.379842e-01  2.820316e-01
18  checkoutservice         mem     25  1.211242e+07  1.226118e+07  1.227127e+07
19  checkoutservice      socket     25  9.000000e+00  9.000000e+00  9.000000e+00```
```

In [ ]:
# In[2]:

import pandas as pd

# reuse metric if available otherwise load
if 'metric' not in globals():
    metric = pd.read_csv('metric.csv')
    metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
    metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
else:
    # ensure types are correct
    if metric['timestamp'].dtype == 'int64' or metric['timestamp'].dtype == 'float64':
        metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
    metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# 1) compute global P95 and P05 per cmdb_id + kpi_name using full series (no filtering)
global_thr = (
    metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(p95=lambda x: x.quantile(0.95), p05=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) filter metric to the incident window (UTC)
start = pd.to_datetime("2024-01-17 22:16:32", utc=True)
end   = pd.to_datetime("2024-01-17 22:46:32", utc=True)
window_df = metric[(metric['timestamp'] >= start) & (metric['timestamp'] <= end)].copy()

# If window is empty, produce an empty summary with expected columns
if window_df.shape[0] == 0:
    summary = pd.DataFrame(columns=[
        'cmdb_id', 'kpi_name', 'total_points_in_window',
        'high_anomaly_count', 'earliest_high_anomaly_timestamp_utc', 'max_value_in_window',
        'low_anomaly_count', 'earliest_low_anomaly_timestamp_utc', 'min_value_in_window'
    ])
else:
    # 3) attach global thresholds to each row in the window
    window_df = window_df.merge(global_thr, on=['cmdb_id', 'kpi_name'], how='left')

    # 4) mark anomalies
    window_df['is_high'] = window_df['value'] >= window_df['p95']
    window_df['is_low']  = window_df['value'] <= window_df['p05']

    # create masked timestamp columns for earliest detection
    window_df['ts_if_high'] = window_df['timestamp'].where(window_df['is_high'])
    window_df['ts_if_low']  = window_df['timestamp'].where(window_df['is_low'])

    # 5) aggregate per cmdb_id + kpi_name within the window
    agg = window_df.groupby(['cmdb_id', 'kpi_name']).agg(
        total_points_in_window = ('value', 'size'),
        high_anomaly_count = ('is_high', 'sum'),
        earliest_high_anomaly_timestamp = ('ts_if_high', 'min'),
        max_value_in_window = ('value', 'max'),
        low_anomaly_count = ('is_low', 'sum'),
        earliest_low_anomaly_timestamp = ('ts_if_low', 'min'),
        min_value_in_window = ('value', 'min'),
    ).reset_index()

    # format earliest timestamps as UTC strings; replace NaT with None (null)
    def fmt_ts(s):
        return s.dt.strftime('%Y-%m-%d %H:%M:%S %Z').where(s.notna())

    agg['earliest_high_anomaly_timestamp_utc'] = fmt_ts(agg['earliest_high_anomaly_timestamp'])
    agg['earliest_low_anomaly_timestamp_utc']  = fmt_ts(agg['earliest_low_anomaly_timestamp'])

    # drop intermediate timestamp columns
    summary = agg[
        ['cmdb_id', 'kpi_name', 'total_points_in_window',
         'high_anomaly_count', 'earliest_high_anomaly_timestamp_utc', 'max_value_in_window',
         'low_anomaly_count', 'earliest_low_anomaly_timestamp_utc', 'min_value_in_window']
    ]

    # replace NaN strings with None to represent nulls
    summary['earliest_high_anomaly_timestamp_utc'] = summary['earliest_high_anomaly_timestamp_utc'].where(
        summary['earliest_high_anomaly_timestamp_utc'].notna(), None
    )
    summary['earliest_low_anomaly_timestamp_utc'] = summary['earliest_low_anomaly_timestamp_utc'].where(
        summary['earliest_low_anomaly_timestamp_utc'].notna(), None
    )

    # 6) sort by high_anomaly_count desc, then low_anomaly_count desc, then total_points_in_window desc
    summary = summary.sort_values(
        ['high_anomaly_count', 'low_anomaly_count', 'total_points_in_window'],
        ascending=[False, False, False]
    ).head(20)

summary

```
Out[2]:
```
summary = (
    "Analysis summary for window 2024-01-17 22:16:32 — 2024-01-17 22:46:32 UTC:\n\n"
    "1) Many cmdb_id + kpi combinations had every point in the window (25 points) flagged as both high (>= global P95) and low (<= global P05). "
    "Those include: cartservice.diskio, cartservice.socket, checkoutservice.diskio, checkoutservice.socket, currencyservice.socket, "
    "emailservice.diskio, emailservice.socket, paymentservice.diskio, paymentservice.socket, productcatalogservice.error, recommendationservice.diskio, "
    "recommendationservice.socket, redis.socket, shippingservice.socket. Earliest anomaly timestamps for these are 2024-01-17 22:19:00 UTC.\n\n"
    "2) Notable deviations from the above pattern:\n"
    "- adservice.socket: 25 high anomalies, 24 low anomalies (earliest at 2024-01-17 22:19:00 UTC; max ≈ 4.233333, min = 4.0).\n"
    "- frontend.socket: 6 high anomalies (earliest high 2024-01-17 22:38:00 UTC), 5 low anomalies (earliest low 2024-01-17 22:26:00 UTC); max = 33, min = 25.\n"
    "- emailservice.mem: 5 high anomalies (earliest high 22:38:00 UTC), 2 low anomalies (earliest low 22:23:00 UTC); values ~4.358e7.\n"
    "- productcatalogservice.socket: 2 high anomalies (earliest 22:33:00 UTC), 23 low anomalies (earliest 22:19:00 UTC); max = 3.3, min = 3.0.\n\n"
    "3) All summarized rows show total_points_in_window = 25. Maximums and minimums vary by KPI (e.g., frontend.socket max=33, many diskio metrics are 0). \n\n"
    "Conclusion: The window contains many metrics where every sample is classified as both high and low relative to the global P95/P05 thresholds, plus a few KPIs with mixed counts. "
    "Refer to the detailed summary table for exact counts and earliest timestamps per cmdb_id + kpi_name."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id kpi_name  total_points_in_window  high_anomaly_count earliest_high_anomaly_timestamp_utc  max_value_in_window  low_anomaly_count earliest_low_anomaly_timestamp_utc  min_value_in_window
8             cartservice   diskio                      25                  25             2024-01-17 22:19:00 UTC         0.000000e+00                 25            2024-01-17 22:19:00 UTC         0.000000e+00
12            cartservice   socket                      25                  25             2024-01-17 22:19:00 UTC         4.000000e+00                 25            2024-01-17 22:19:00 UTC         4.000000e+00
15        checkoutservice   diskio                      25                  25             2024-01-17 22:19:00 UTC         0.000000e+00                 25            2024-01-17 22:19:00 UTC         0.000000e+00
19        checkoutservice   socket                      25                  25             2024-01-17 22:19:00 UTC         9.000000e+00                 25            2024-01-17 22:19:00 UTC         9.000000e+00
25        currencyservice   socket                      25                  25             2024-01-17 22:19:00 UTC         4.000000e+00                 25            2024-01-17 22:19:00 UTC         4.000000e+00
28           emailservice   diskio                      25                  25             2024-01-17 22:19:00 UTC         0.000000e+00                 25            2024-01-17 22:19:00 UTC         0.000000e+00
32           emailservice   socket                      25                  25             2024-01-17 22:19:00 UTC         3.000000e+00                 25            2024-01-17 22:19:00 UTC         3.000000e+00
44         paymentservice   diskio                      25                  25             2024-01-17 22:19:00 UTC         0.000000e+00                 25            2024-01-17 22:19:00 UTC         0.000000e+00
48         paymentservice   socket                      25                  25             2024-01-17 22:19:00 UTC         3.000000e+00                 25            2024-01-17 22:19:00 UTC         3.000000e+00
51  productcatalogservice    error                      25                  25             2024-01-17 22:19:00 UTC         0.000000e+00                 25            2024-01-17 22:19:00 UTC         0.000000e+00
58  recommendationservice   diskio                      25                  25             2024-01-17 22:19:00 UTC         0.000000e+00                 25            2024-01-17 22:19:00 UTC         0.000000e+00
62  recommendationservice   socket                      25                  25             2024-01-17 22:19:00 UTC         4.000000e+00                 25            2024-01-17 22:19:00 UTC         4.000000e+00
67                  redis   socket                      25                  25             2024-01-17 22:19:00 UTC         4.000000e+00                 25            2024-01-17 22:19:00 UTC         4.000000e+00
72        shippingservice   socket                      25                  25             2024-01-17 22:19:00 UTC         2.000000e+00                 25            2024-01-17 22:19:00 UTC         2.000000e+00
5               adservice   socket                      25                  25             2024-01-17 22:19:00 UTC         4.233333e+00                 24            2024-01-17 22:19:00 UTC         4.000000e+00
35               frontend    error                      25                  25             2024-01-17 22:19:00 UTC         1.675000e-02                 24            2024-01-17 22:19:00 UTC         0.000000e+00
41      frontend-external    error                      25                  25             2024-01-17 22:19:00 UTC         3.325000e-02                 24            2024-01-17 22:19:00 UTC         0.000000e+00
39               frontend   socket                      25                   6             2024-01-17 22:38:00 UTC         3.300000e+01                  5            2024-01-17 22:26:00 UTC         2.500000e+01
31           emailservice      mem                      25                   5             2024-01-17 22:38:00 UTC         4.358670e+07                  2            2024-01-17 22:23:00 UTC         4.358014e+07
55  productcatalogservice   socket                      25                   2             2024-01-17 22:33:00 UTC         3.300000e+00                 23            2024-01-17 22:19:00 UTC         3.000000e+00```
```

In [ ]:
# In[3]:

import pandas as pd

# load trace.csv (reuse variables if available)
if 'trace' not in globals():
    trace = pd.read_csv('trace.csv')
    trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
    trace['value'] = pd.to_numeric(trace['value'], errors='coerce')
else:
    if trace['timestamp'].dtype != 'datetime64[ns, UTC]':
        trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
    trace['value'] = pd.to_numeric(trace['value'], errors='coerce')

# 1) compute global P95 per cmdb_id + trace_name using the full series
global_p95 = (
    trace
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# 2) filter trace to the incident window (reuse start/end if available)
if 'start' not in globals() or 'end' not in globals():
    start = pd.to_datetime("2024-01-17 22:16:32", utc=True)
    end   = pd.to_datetime("2024-01-17 22:46:32", utc=True)

window_trace = trace[(trace['timestamp'] >= start) & (trace['timestamp'] <= end)].copy()

# 3) if window empty, return empty summary with expected columns
if window_trace.shape[0] == 0:
    trace_summary = pd.DataFrame(columns=[
        'cmdb_id', 'trace_name', 'total_points_in_window',
        'high_anomaly_count', 'earliest_high_anomaly_timestamp_utc', 'max_value_in_window'
    ])
else:
    # attach global p95
    window_trace = window_trace.merge(global_p95, on=['cmdb_id', 'trace_name'], how='left')

    # mark high anomalies
    window_trace['is_high'] = window_trace['value'] >= window_trace['p95']
    window_trace['ts_if_high'] = window_trace['timestamp'].where(window_trace['is_high'])

    # aggregate per cmdb_id + trace_name within the window
    agg = window_trace.groupby(['cmdb_id', 'trace_name']).agg(
        total_points_in_window = ('value', 'size'),
        high_anomaly_count = ('is_high', 'sum'),
        earliest_high_anomaly_timestamp = ('ts_if_high', 'min'),
        max_value_in_window = ('value', 'max'),
    ).reset_index()

    # format earliest timestamp string or null
    agg['earliest_high_anomaly_timestamp_utc'] = (
        agg['earliest_high_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
        .where(agg['earliest_high_anomaly_timestamp'].notna())
    )
    agg['earliest_high_anomaly_timestamp_utc'] = agg['earliest_high_anomaly_timestamp_utc'].where(
        agg['earliest_high_anomaly_timestamp_utc'].notna(), None
    )

    trace_summary = agg[
        ['cmdb_id', 'trace_name', 'total_points_in_window',
         'high_anomaly_count', 'earliest_high_anomaly_timestamp_utc', 'max_value_in_window']
    ]

    # sort as requested and keep top 20
    trace_summary = trace_summary.sort_values(
        ['high_anomaly_count', 'total_points_in_window'],
        ascending=[False, False]
    ).head(20)

trace_summary

```
Out[3]:
```
summary = (
    "Summary for traces in 2024-01-17 22:16:32 — 2024-01-17 22:46:32 UTC:\n\n"
    "- All listed cmdb_id + trace_name rows have 25 samples in the window and 25 high anomalies "
    "(value >= global P95).\n"
    "- Earliest high anomaly for all rows is 2024-01-17 22:19:00 UTC.\n"
    "- The reported max value_in_window for every listed trace error_rate metric is 0.0.\n\n"
    "Interpretation: many trace error_rate metrics (checkoutservice, currencyservice, emailservice, frontendservice and their "
    "calls) were classified as at-or-above their global P95 for every sample in the window, but the absolute values are 0.0. "
    "This indicates the global P95 threshold for these error_rate traces is 0.0 and no non-zero error rates were observed in the window. "
    "Recommend reviewing the global threshold calculation and examining raw traces if you expect non-zero error rates."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                 trace_name  total_points_in_window  high_anomaly_count earliest_high_anomaly_timestamp_utc  max_value_in_window
2   checkoutservice      trace.from_checkoutservice.error_rate                      25                  25             2024-01-17 22:19:00 UTC                  0.0
6   checkoutservice      trace.from_frontendservice.error_rate                      25                  25             2024-01-17 22:19:00 UTC                  0.0
10  checkoutservice                 trace.from_root.error_rate                      25                  25             2024-01-17 22:19:00 UTC                  0.0
14  checkoutservice        trace.to_checkoutservice.error_rate                      25                  25             2024-01-17 22:19:00 UTC                  0.0
18  checkoutservice        trace.to_currencyservice.error_rate                      25                  25             2024-01-17 22:19:00 UTC                  0.0
22  checkoutservice           trace.to_emailservice.error_rate                      25                  25             2024-01-17 22:19:00 UTC                  0.0
26  checkoutservice         trace.to_paymentservice.error_rate                      25                  25             2024-01-17 22:19:00 UTC                  0.0
30  checkoutservice  trace.to_productcatalogservice.error_rate                      25                  25             2024-01-17 22:19:00 UTC                  0.0
34  currencyservice      trace.from_checkoutservice.error_rate                      25                  25             2024-01-17 22:19:00 UTC                  0.0
38  currencyservice      trace.from_frontendservice.error_rate                      25                  25             2024-01-17 22:19:00 UTC                  0.0
42  currencyservice                 trace.from_root.error_rate                      25                  25             2024-01-17 22:19:00 UTC                  0.0
46     emailservice      trace.from_checkoutservice.error_rate                      25                  25             2024-01-17 22:19:00 UTC                  0.0
50     emailservice                 trace.from_root.error_rate                      25                  25             2024-01-17 22:19:00 UTC                  0.0
54  frontendservice      trace.from_frontendservice.error_rate                      25                  25             2024-01-17 22:19:00 UTC                  0.0
58  frontendservice                 trace.from_root.error_rate                      25                  25             2024-01-17 22:19:00 UTC                  0.0
62  frontendservice        trace.to_checkoutservice.error_rate                      25                  25             2024-01-17 22:19:00 UTC                  0.0
66  frontendservice        trace.to_currencyservice.error_rate                      25                  25             2024-01-17 22:19:00 UTC                  0.0
70  frontendservice        trace.to_frontendservice.error_rate                      25                  25             2024-01-17 22:19:00 UTC                  0.0
74  frontendservice  trace.to_productcatalogservice.error_rate                      25                  25             2024-01-17 22:19:00 UTC                  0.0
78  frontendservice  trace.to_recommendationservice.error_rate                      25                  25             2024-01-17 22:19:00 UTC                  0.0```
```

In [ ]:
# In[4]:

import pandas as pd

# load log.csv if not already loaded
if 'log' not in globals():
    log = pd.read_csv('log.csv')

# parse timestamp as UTC and ensure numeric value
if log['timestamp'].dtype != 'datetime64[ns, UTC]':
    log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)
log['value'] = pd.to_numeric(log['value'], errors='coerce')

# incident window (reuse start/end if present, otherwise define)
if 'start' not in globals() or 'end' not in globals():
    start = pd.to_datetime("2024-01-17 22:16:32", utc=True)
    end   = pd.to_datetime("2024-01-17 22:46:32", utc=True)

# filter to window
log_window = log[(log['timestamp'] >= start) & (log['timestamp'] <= end)].copy()

# if empty, prepare empty summary dataframe with expected columns
if log_window.shape[0] == 0:
    log_summary = pd.DataFrame(columns=[
        'cmdb_id', 'log_name', 'total_points_in_window',
        'max_value_in_window', 'earliest_timestamp_utc'
    ])
else:
    agg = (
        log_window
        .groupby(['cmdb_id', 'log_name'])
        .agg(
            total_points_in_window = ('value', 'size'),
            max_value_in_window = ('value', 'max'),
            earliest_timestamp = ('timestamp', 'min')
        )
        .reset_index()
    )

    # format earliest timestamp as UTC string, replace NaT with None
    agg['earliest_timestamp_utc'] = agg['earliest_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    agg['earliest_timestamp_utc'] = agg['earliest_timestamp_utc'].where(agg['earliest_timestamp'].notna(), None)

    log_summary = agg[
        ['cmdb_id', 'log_name', 'total_points_in_window', 'max_value_in_window', 'earliest_timestamp_utc']
    ].sort_values(
        ['total_points_in_window', 'cmdb_id'],
        ascending=[False, True]
    ).head(20)

log_summary

```
Out[4]:
```
summary = (
    "Log summary for 2024-01-17 22:16:32 — 2024-01-17 22:46:32 UTC:\n\n"
    "- Most services have 25 log samples in the window; redis has 5 samples.\n"
    "- For every service, log.error_count max = 0 (no error-counts observed in this window).\n"
    "- log.row_count (max per service) indicates relative traffic: frontend=2777, currencyservice=2491, cartservice=803, "
    "recommendationservice=588, shippingservice=486, adservice=460, checkoutservice=111, paymentservice=74, emailservice=37, redis=5.\n"
    "- Earliest log timestamps are 2024-01-17 22:19:00 UTC for most services; redis earliest is 2024-01-17 22:23:00 UTC.\n\n"
    "Interpretation: no error counts were recorded in logs during the incident window, and frontend/currencyservice show the highest row counts."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id         log_name  total_points_in_window  max_value_in_window   earliest_timestamp_utc
0               adservice  log.error_count                      25                    0  2024-01-17 22:19:00 UTC
1               adservice    log.row_count                      25                  460  2024-01-17 22:19:00 UTC
2             cartservice  log.error_count                      25                    0  2024-01-17 22:19:00 UTC
3             cartservice    log.row_count                      25                  803  2024-01-17 22:19:00 UTC
4         checkoutservice  log.error_count                      25                    0  2024-01-17 22:19:00 UTC
5         checkoutservice    log.row_count                      25                  111  2024-01-17 22:19:00 UTC
6         currencyservice  log.error_count                      25                    0  2024-01-17 22:19:00 UTC
7         currencyservice    log.row_count                      25                 2491  2024-01-17 22:19:00 UTC
8            emailservice  log.error_count                      25                    0  2024-01-17 22:19:00 UTC
9            emailservice    log.row_count                      25                   37  2024-01-17 22:19:00 UTC
10               frontend  log.error_count                      25                    0  2024-01-17 22:19:00 UTC
11               frontend    log.row_count                      25                 2777  2024-01-17 22:19:00 UTC
12         paymentservice  log.error_count                      25                    0  2024-01-17 22:19:00 UTC
13         paymentservice    log.row_count                      25                   74  2024-01-17 22:19:00 UTC
14  recommendationservice  log.error_count                      25                    0  2024-01-17 22:19:00 UTC
15  recommendationservice    log.row_count                      25                  588  2024-01-17 22:19:00 UTC
18        shippingservice  log.error_count                      25                    0  2024-01-17 22:19:00 UTC
19        shippingservice    log.row_count                      25                  486  2024-01-17 22:19:00 UTC
16                  redis  log.error_count                       5                    0  2024-01-17 22:23:00 UTC
17                  redis    log.row_count                       5                    5  2024-01-17 22:23:00 UTC```
```

In [ ]:
# In[5]:

import pandas as pd

# ensure trace is loaded and types are correct (reuse if available)
if 'trace' not in globals():
    trace = pd.read_csv('trace.csv')
    trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
    trace['value'] = pd.to_numeric(trace['value'], errors='coerce')
else:
    # ensure timestamp is parsed and value numeric
    if trace['timestamp'].dtype != 'datetime64[ns, UTC]':
        trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
    trace['value'] = pd.to_numeric(trace['value'], errors='coerce')

# 1) compute global P95 per cmdb_id + trace_name using the full available series (no filtering)
global_p95 = (
    trace
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# 2) incident window (reuse if available)
if 'start' not in globals() or 'end' not in globals():
    start = pd.to_datetime("2024-01-17 22:16:32", utc=True)
    end   = pd.to_datetime("2024-01-17 22:46:32", utc=True)

# 3) filter trace to the window and to trace_name containing 'duration' or 'latency' (case-insensitive)
pattern = r'duration|latency'
window_trace = trace[
    (trace['timestamp'] >= start) &
    (trace['timestamp'] <= end) &
    (trace['trace_name'].str.contains(pattern, case=False, na=False))
].copy()

# 4) prepare summary
if window_trace.shape[0] == 0:
    trace_summary = pd.DataFrame(columns=[
        'cmdb_id', 'trace_name', 'total_points_in_window',
        'high_anomaly_count', 'earliest_high_anomaly_timestamp_utc', 'max_value_in_window'
    ])
else:
    # attach global p95 thresholds
    window_trace = window_trace.merge(global_p95, on=['cmdb_id', 'trace_name'], how='left')

    # mark high anomalies where value >= global p95
    window_trace['is_high'] = window_trace['value'] >= window_trace['p95']
    window_trace['ts_if_high'] = window_trace['timestamp'].where(window_trace['is_high'])

    agg = window_trace.groupby(['cmdb_id', 'trace_name']).agg(
        total_points_in_window = ('value', 'size'),
        high_anomaly_count = ('is_high', 'sum'),
        earliest_high_anomaly_timestamp = ('ts_if_high', 'min'),
        max_value_in_window = ('value', 'max'),
    ).reset_index()

    # format earliest timestamps as UTC string, replace NaT with None
    agg['earliest_high_anomaly_timestamp_utc'] = (
        agg['earliest_high_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
        .where(agg['earliest_high_anomaly_timestamp'].notna())
    )
    agg['earliest_high_anomaly_timestamp_utc'] = agg['earliest_high_anomaly_timestamp_utc'].where(
        agg['earliest_high_anomaly_timestamp_utc'].notna(), None
    )

    trace_summary = agg[
        ['cmdb_id', 'trace_name', 'total_points_in_window',
         'high_anomaly_count', 'earliest_high_anomaly_timestamp_utc', 'max_value_in_window']
    ].sort_values(['high_anomaly_count', 'max_value_in_window'], ascending=[False, False]).head(20)

trace_summary

```
Out[5]:
```
summary = (
    "Trace duration/latency summary for 2024-01-17 22:16:32–22:46:32 UTC:\n\n"
    "- All listed trace series have 25 samples in the window.\n"
    "- Highest counts of samples >= global P95 (4 samples):\n"
    "  • productcatalogservice.trace.from_recommendationservice.duration_p95 (earliest high 22:19:00 UTC, max=0.000033s)\n"
    "  • recommendationservice.trace.to_productcatalogservice.duration_p95 (earliest high 22:19:00 UTC, max=0.000033s)\n"
    "- Three-sample hits (earliest high ~22:21) occur for frontend <-> productcatalog p95-duration traces (max ≈ 0.000029s).\n"
    "- Several traces have 2 high samples; of these, the largest absolute durations are:\n"
    "  • trace.from_root/to_frontend.duration_p95 (max ≈ 1.90361 s, earliest high 22:39:00 UTC)\n"
    "  • trace.from_root/to_frontend.duration_mean (max ≈ 0.385186 s, earliest high 22:37:00 UTC)\n"
    "  • frontend <-> checkout and frontend <-> recommendation duration_p95/mean traces (max ≈ 0.3035s and ≈0.1227s, earliest highs ~22:24–22:37 UTC).\n\n"
    "Interpretation: many p95-duration anomalies are very small in absolute terms (microseconds) and likely not impactful. However, the from_root/to_frontend and some duration_mean traces show noticeably larger absolute latencies (≈0.38s–1.9s) with earliest high samples around 22:37–22:39 UTC — those paths (root ↔ frontend, and frontend ↔ productcatalog/recommendation) warrant investigation."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                      trace_name  total_points_in_window  high_anomaly_count earliest_high_anomaly_timestamp_utc  max_value_in_window
49  productcatalogservice   trace.from_recommendationservice.duration_p95                      25                   4             2024-01-17 22:19:00 UTC             0.000033
59  recommendationservice     trace.to_productcatalogservice.duration_p95                      25                   4             2024-01-17 22:19:00 UTC             0.000033
37        frontendservice     trace.to_productcatalogservice.duration_p95                      25                   3             2024-01-17 22:21:00 UTC             0.000029
47  productcatalogservice         trace.from_frontendservice.duration_p95                      25                   3             2024-01-17 22:21:00 UTC             0.000029
29        frontendservice                    trace.from_root.duration_p95                      25                   2             2024-01-17 22:39:00 UTC             1.903610
69                   root           trace.to_frontendservice.duration_p95                      25                   2             2024-01-17 22:39:00 UTC             1.903610
28        frontendservice                   trace.from_root.duration_mean                      25                   2             2024-01-17 22:37:00 UTC             0.385186
68                   root          trace.to_frontendservice.duration_mean                      25                   2             2024-01-17 22:37:00 UTC             0.385186
3         checkoutservice         trace.from_frontendservice.duration_p95                      25                   2             2024-01-17 22:37:00 UTC             0.303522
31        frontendservice           trace.to_checkoutservice.duration_p95                      25                   2             2024-01-17 22:37:00 UTC             0.303522
39        frontendservice     trace.to_recommendationservice.duration_p95                      25                   2             2024-01-17 22:37:00 UTC             0.298557
53  recommendationservice         trace.from_frontendservice.duration_p95                      25                   2             2024-01-17 22:37:00 UTC             0.298557
55  recommendationservice   trace.from_recommendationservice.duration_p95                      25                   2             2024-01-17 22:40:00 UTC             0.283881
61  recommendationservice     trace.to_recommendationservice.duration_p95                      25                   2             2024-01-17 22:40:00 UTC             0.283881
2         checkoutservice        trace.from_frontendservice.duration_mean                      25                   2             2024-01-17 22:24:00 UTC             0.139947
30        frontendservice          trace.to_checkoutservice.duration_mean                      25                   2             2024-01-17 22:24:00 UTC             0.139947
38        frontendservice    trace.to_recommendationservice.duration_mean                      25                   2             2024-01-17 22:37:00 UTC             0.122722
52  recommendationservice        trace.from_frontendservice.duration_mean                      25                   2             2024-01-17 22:37:00 UTC             0.122722
54  recommendationservice  trace.from_recommendationservice.duration_mean                      25                   2             2024-01-17 22:37:00 UTC             0.115261
60  recommendationservice    trace.to_recommendationservice.duration_mean                      25                   2             2024-01-17 22:37:00 UTC             0.115261```
```

In [ ]:
# In[6]:

import pandas as pd

# reuse metric and start/end if available, otherwise load/define
if 'metric' not in globals():
    metric = pd.read_csv('metric.csv')
    metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
    metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
else:
    # ensure types are correct
    if metric['timestamp'].dtype != 'datetime64[ns, UTC]':
        metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
    metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

if 'start' not in globals() or 'end' not in globals():
    start = pd.to_datetime("2024-01-17 22:16:32", utc=True)
    end   = pd.to_datetime("2024-01-17 22:46:32", utc=True)

# restrict to the requested components
components = [
    "checkoutservice", "currencyservice", "emailservice",
    "productcatalogservice", "recommendationservice"
]

metric_sub = metric[metric['cmdb_id'].isin(components)].copy()

# 1) compute global P95 and P05 per cmdb_id + kpi_name using the full available series (before window filtering)
global_thr = (
    metric_sub
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(p95=lambda x: x.quantile(0.95), p05=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) filter metric to the incident window
window_df = metric_sub[(metric_sub['timestamp'] >= start) & (metric_sub['timestamp'] <= end)].copy()

# prepare empty summary if no data in window
if window_df.shape[0] == 0:
    summary = pd.DataFrame(columns=[
        'cmdb_id', 'kpi_name', 'total_points_in_window',
        'high_anomaly_count', 'earliest_high_anomaly_timestamp_utc',
        'low_anomaly_count', 'earliest_low_anomaly_timestamp_utc',
        'max_value_in_window', 'min_value_in_window'
    ])
else:
    # attach global thresholds (calculated from full series) to window rows
    window_df = window_df.merge(global_thr, on=['cmdb_id', 'kpi_name'], how='left')

    # mark anomalies
    window_df['is_high'] = window_df['value'] >= window_df['p95']
    window_df['is_low']  = window_df['value'] <= window_df['p05']

    # helper timestamp columns for earliest detection
    window_df['ts_if_high'] = window_df['timestamp'].where(window_df['is_high'])
    window_df['ts_if_low']  = window_df['timestamp'].where(window_df['is_low'])

    # aggregate per component + kpi within the window
    agg = window_df.groupby(['cmdb_id', 'kpi_name']).agg(
        total_points_in_window = ('value', 'size'),
        high_anomaly_count = ('is_high', 'sum'),
        earliest_high_anomaly_timestamp = ('ts_if_high', 'min'),
        low_anomaly_count = ('is_low', 'sum'),
        earliest_low_anomaly_timestamp = ('ts_if_low', 'min'),
        max_value_in_window = ('value', 'max'),
        min_value_in_window = ('value', 'min'),
    ).reset_index()

    # format earliest timestamps as UTC strings, replace NaT with None
    agg['earliest_high_anomaly_timestamp_utc'] = (
        agg['earliest_high_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
        .where(agg['earliest_high_anomaly_timestamp'].notna())
    )
    agg['earliest_high_anomaly_timestamp_utc'] = agg['earliest_high_anomaly_timestamp_utc'].where(
        agg['earliest_high_anomaly_timestamp_utc'].notna(), None
    )

    agg['earliest_low_anomaly_timestamp_utc'] = (
        agg['earliest_low_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
        .where(agg['earliest_low_anomaly_timestamp'].notna())
    )
    agg['earliest_low_anomaly_timestamp_utc'] = agg['earliest_low_anomaly_timestamp_utc'].where(
        agg['earliest_low_anomaly_timestamp_utc'].notna(), None
    )

    # select and sort as requested
    summary = agg[
        ['cmdb_id', 'kpi_name', 'total_points_in_window',
         'high_anomaly_count', 'earliest_high_anomaly_timestamp_utc',
         'low_anomaly_count', 'earliest_low_anomaly_timestamp_utc',
         'max_value_in_window', 'min_value_in_window']
    ].sort_values(['high_anomaly_count', 'low_anomaly_count'], ascending=[False, False]).head(20)

# display the compact summary table
summary

```
Out[6]:
```
summary = (
    "Summary for the five components (checkoutservice, currencyservice, emailservice, "
    "productcatalogservice, recommendationservice) during 2024-01-17 22:16:32–22:46:32 UTC:\n\n"
    "1) Many KPIs (diskio, socket, error) show every sample in the window flagged as BOTH high (>= global P95) and low (<= global P05). "
    "For these metrics the values are constant (e.g., diskio = 0, error = 0, socket = 3/4/9), so P05 and P95 align with the observed values. "
    "This likely reflects zero variance or thresholds equal to observed values rather than meaningful anomalies.\n\n"
    "2) Emailservice.mem shows a small number of high samples (5) and 2 low samples: max ≈ 4.35867e7, min ≈ 4.358014e7. "
    "Earliest low at 2024-01-17 22:23:00 UTC and earliest high at 22:38:00 UTC — indicating a late-window upward shift worth investigating.\n\n"
    "3) Productcatalogservice.socket has 2 high samples (earliest 22:33:00 UTC) and 23 low samples (earliest 22:19:00 UTC); "
    "values range 3.0–3.3, so most points are at the lower value with a brief higher value around 22:33.\n\n"
    "4) Several latency/cpu/memory/workload KPIs for checkoutservice and currencyservice show small, balanced counts (typically 2 high and 2 low) "
    "with modest max/min ranges (example: checkoutservice latency-90 max ≈ 0.282s, min ≈ 0.210s). These indicate modest variability but no large sustained spike.\n\n"
    "Recommendation: Investigate the emailservice memory increase around 22:38 and the productcatalogservice socket uptick at ~22:33. "
    "Also verify the global P05/P95 calculations for the many constant-value metrics (diskio, error, socket) because thresholds that equal observed values will mark every point as both high and low and can be misleading."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  total_points_in_window  high_anomaly_count earliest_high_anomaly_timestamp_utc  low_anomaly_count earliest_low_anomaly_timestamp_utc  max_value_in_window  min_value_in_window
1         checkoutservice      diskio                      25                  25             2024-01-17 22:19:00 UTC                 25            2024-01-17 22:19:00 UTC         0.000000e+00         0.000000e+00
5         checkoutservice      socket                      25                  25             2024-01-17 22:19:00 UTC                 25            2024-01-17 22:19:00 UTC         9.000000e+00         9.000000e+00
11        currencyservice      socket                      25                  25             2024-01-17 22:19:00 UTC                 25            2024-01-17 22:19:00 UTC         4.000000e+00         4.000000e+00
14           emailservice      diskio                      25                  25             2024-01-17 22:19:00 UTC                 25            2024-01-17 22:19:00 UTC         0.000000e+00         0.000000e+00
18           emailservice      socket                      25                  25             2024-01-17 22:19:00 UTC                 25            2024-01-17 22:19:00 UTC         3.000000e+00         3.000000e+00
21  productcatalogservice       error                      25                  25             2024-01-17 22:19:00 UTC                 25            2024-01-17 22:19:00 UTC         0.000000e+00         0.000000e+00
28  recommendationservice      diskio                      25                  25             2024-01-17 22:19:00 UTC                 25            2024-01-17 22:19:00 UTC         0.000000e+00         0.000000e+00
32  recommendationservice      socket                      25                  25             2024-01-17 22:19:00 UTC                 25            2024-01-17 22:19:00 UTC         4.000000e+00         4.000000e+00
17           emailservice         mem                      25                   5             2024-01-17 22:38:00 UTC                  2            2024-01-17 22:23:00 UTC         4.358670e+07         4.358014e+07
25  productcatalogservice      socket                      25                   2             2024-01-17 22:33:00 UTC                 23            2024-01-17 22:19:00 UTC         3.300000e+00         3.000000e+00
15           emailservice  latency-50                      25                   2             2024-01-17 22:26:00 UTC                  6            2024-01-17 22:22:00 UTC         3.159378e-03         3.000000e-03
16           emailservice  latency-90                      25                   2             2024-01-17 22:26:00 UTC                  6            2024-01-17 22:22:00 UTC         5.307841e-03         4.600000e-03
0         checkoutservice         cpu                      25                   2             2024-01-17 22:19:00 UTC                  2            2024-01-17 22:33:00 UTC         5.511535e-01         3.304260e-01
2         checkoutservice  latency-50                      25                   2             2024-01-17 22:19:00 UTC                  2            2024-01-17 22:30:00 UTC         1.226884e-01         6.205051e-02
3         checkoutservice  latency-90                      25                   2             2024-01-17 22:38:00 UTC                  2            2024-01-17 22:20:00 UTC         2.820316e-01         2.098486e-01
4         checkoutservice         mem                      25                   2             2024-01-17 22:31:00 UTC                  2            2024-01-17 22:19:00 UTC         1.227127e+07         1.181989e+07
6         checkoutservice    workload                      25                   2             2024-01-17 22:19:00 UTC                  2            2024-01-17 22:26:00 UTC         1.238143e+00         6.356333e-01
7         currencyservice         cpu                      25                   2             2024-01-17 22:19:00 UTC                  2            2024-01-17 22:41:00 UTC         3.368379e+01         2.822614e+01
8         currencyservice  latency-50                      25                   2             2024-01-17 22:19:00 UTC                  2            2024-01-17 22:42:00 UTC         3.872463e-03         3.451278e-03
9         currencyservice  latency-90                      25                   2             2024-01-17 22:19:00 UTC                  2            2024-01-17 22:40:00 UTC         4.384771e-02         1.529553e-02```
```